In [2]:
import pandas as pd

In [3]:
base_df = pd.read_csv(
    "../data/processed/caba_apartments_base.csv"
)

clean_df = pd.read_csv(
    "../data/processed/caba_apartments_clean.csv"
)

In [4]:
print(base_df.shape)
print(clean_df.shape)

(121442, 25)
(96735, 26)


In [5]:
powerbi_df = base_df.copy()

In [6]:
powerbi_df["end_date_clean"] = powerbi_df["end_date"].replace(
    "9999-12-31",
    pd.NA
)

powerbi_df["start_date"] = pd.to_datetime(
    powerbi_df["start_date"]
)

powerbi_df["end_date_clean"] = pd.to_datetime(
    powerbi_df["end_date_clean"]
)

powerbi_df["listing_duration_days"] = (
    powerbi_df["end_date_clean"] - powerbi_df["start_date"]
).dt.days

In [7]:
powerbi_df[
    ["start_date", "end_date", "end_date_clean", "listing_duration_days"]
].head(10)

,start_date,end_date,end_date_clean,listing_duration_days
0,2019-09-15,9999-12-31,NaT,NaN
1,2019-09-15,9999-12-31,NaT,NaN
2,2019-09-15,9999-12-31,NaT,NaN
3,2019-09-15,2019-09-25,2019-09-25,10.0
4,2019-09-15,2019-09-30,2019-09-30,15.0
5,2019-09-15,9999-12-31,NaT,NaN
6,2019-09-15,2020-01-16,2020-01-16,123.0
7,2019-09-15,9999-12-31,NaT,NaN
8,2019-09-15,9999-12-31,NaT,NaN
9,2019-09-15,2019-10-18,2019-10-18,33.0


In [8]:
valid_surface = (
    powerbi_df["surface_total"].notna() &
    (powerbi_df["surface_total"] > 0)
)

In [9]:
powerbi_df["price_per_m2"] = pd.NA

In [10]:
powerbi_df.loc[
    valid_surface,
    "price_per_m2"
] = (
    powerbi_df.loc[valid_surface, "price"]
    / powerbi_df.loc[valid_surface, "surface_total"]
)

In [11]:
powerbi_df[
    ["price", "surface_total", "price_per_m2"]
].head(10)

,price,surface_total,price_per_m2
0,935000.0,193.0,4844.559585
1,89000.0,31.0,2870.967742
2,132900.0,32.0,4153.125
3,230000.0,88.0,2613.636364
4,108000.0,36.0,3000.0
5,269000.0,80.0,3362.5
6,135000.0,56.0,2410.714286
7,270000.0,60.0,4500.0
8,229000.0,55.0,4163.636364
9,167000.0,60.0,2783.333333


In [12]:
powerbi_df["price_per_m2"].notna().sum()

np.int64(96735)

In [13]:
price_bins = [
    0,
    100000,
    150000,
    200000,
    300000,
    500000,
    float("inf")
]

price_labels = [
    "<100k",
    "100k-150k",
    "150k-200k",
    "200k-300k",
    "300k-500k",
    ">500k"
]

powerbi_df["price_range"] = pd.cut(
    powerbi_df["price"],
    bins=price_bins,
    labels=price_labels
)

In [14]:
powerbi_df[
    ["price", "price_range"]
].head(10)

,price,price_range
0,935000.0,>500k
1,89000.0,<100k
2,132900.0,100k-150k
3,230000.0,200k-300k
4,108000.0,100k-150k
5,269000.0,200k-300k
6,135000.0,100k-150k
7,270000.0,200k-300k
8,229000.0,200k-300k
9,167000.0,150k-200k


In [15]:
powerbi_df["price_range"].value_counts().sort_index()

price_range
<100k        26692
100k-150k    32864
150k-200k    20475
200k-300k    20485
300k-500k    12060
>500k         8866
Name: count, dtype: int64

In [16]:
powerbi_df[
    powerbi_df["price_per_m2"] > 10000
].shape

(169, 29)

In [17]:
powerbi_df[
    powerbi_df["price_per_m2"] > 10000
][
    ["l3", "rooms", "surface_total", "price", "price_per_m2"]
].sort_values(
    "price_per_m2",
    ascending=False
).head(30)

,l3,rooms,surface_total,price,price_per_m2
12659,Palermo,1.0,43.0,32434232.0,754284.465116
95582,San Nicolás,1.0,31.0,10082032.0,325226.83871
101805,Caballito,1.0,41.0,9477000.0,231146.341463
71162,Caballito,1.0,41.0,9477000.0,231146.341463
81796,Caballito,1.0,41.0,9477000.0,231146.341463
118159,Caballito,1.0,41.0,9477000.0,231146.341463
35824,Caballito,1.0,41.0,9477000.0,231146.341463
53801,Caballito,1.0,41.0,8748000.0,213365.853659
28118,Caballito,1.0,41.0,7897000.0,192609.756098
11813,Caballito,1.0,41.0,7897000.0,192609.756098


Para los análisis basados en precio por m² se excluyen listings superiores a USD 10.000/m², ya que representan observaciones extremadamente atípicas (<0,2% de los casos con superficie válida) y presentan un riesgo elevado de errores de precio o superficie. Este umbral es un criterio analítico y no implica que valores superiores sean imposibles en el mercado.

In [18]:
pricing_df = powerbi_df[
    powerbi_df["price_per_m2"].notna() &
    (powerbi_df["price_per_m2"] <= 10000)
].copy()

In [19]:
print(powerbi_df.shape)
print(pricing_df.shape)

(121442, 29)
(96566, 29)


In [20]:
pricing_df["segment_median_price_m2"] = (
    pricing_df
    .groupby(["l3", "rooms"])["price_per_m2"]
    .transform("median")
)

In [21]:
pricing_df["price_vs_segment_pct"] = (
    (
        pricing_df["price_per_m2"]
        - pricing_df["segment_median_price_m2"]
    )
    / pricing_df["segment_median_price_m2"]
    * 100
)

In [22]:
pricing_df[
    [
        "l3",
        "rooms",
        "price_per_m2",
        "segment_median_price_m2",
        "price_vs_segment_pct"
    ]
].head(10)

,l3,rooms,price_per_m2,segment_median_price_m2,price_vs_segment_pct
0,Palermo,4.0,4844.559585,3235.294118,49.740933
1,Palermo,2.0,2870.967742,3253.968254,-11.77026
2,Palermo,2.0,4153.125,3253.968254,27.632622
3,Palermo,3.0,2613.636364,3152.173913,-17.084639
4,Palermo,2.0,3000.0,3253.968254,-7.804878
5,Belgrano,4.0,3362.5,3199.190283,5.10472
6,Floresta,3.0,2410.714286,1959.459459,23.029557
7,Recoleta,1.0,4500.0,3513.513514,28.076923
8,Recoleta,2.0,4163.636364,3333.333333,24.909091
9,Barrio Norte,3.0,2783.333333,2946.428571,-5.535354


In [23]:
pricing_df["segment_size"] = (
    pricing_df
    .groupby(["l3", "rooms"])["id"]
    .transform("count")
)

In [24]:
pricing_df[
    [
        "l3",
        "rooms",
        "price_per_m2",
        "segment_median_price_m2",
        "price_vs_segment_pct",
        "segment_size"
    ]
].head(10)

,l3,rooms,price_per_m2,segment_median_price_m2,price_vs_segment_pct,segment_size
0,Palermo,4.0,4844.559585,3235.294118,49.740933,2544.0
1,Palermo,2.0,2870.967742,3253.968254,-11.77026,4514.0
2,Palermo,2.0,4153.125,3253.968254,27.632622,4514.0
3,Palermo,3.0,2613.636364,3152.173913,-17.084639,4043.0
4,Palermo,2.0,3000.0,3253.968254,-7.804878,4514.0
5,Belgrano,4.0,3362.5,3199.190283,5.10472,2028.0
6,Floresta,3.0,2410.714286,1959.459459,23.029557,211.0
7,Recoleta,1.0,4500.0,3513.513514,28.076923,807.0
8,Recoleta,2.0,4163.636364,3333.333333,24.909091,1153.0
9,Barrio Norte,3.0,2783.333333,2946.428571,-5.535354,1068.0


In [25]:
q1 = pricing_df["price_vs_segment_pct"].quantile(0.25)
q3 = pricing_df["price_vs_segment_pct"].quantile(0.75)

iqr = q3 - q1

upper_limit = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper limit:", upper_limit)

Q1: -12.865910925232958
Q3: 14.580336105845042
IQR: 27.446247031078002
Upper limit: 55.749706652462045


In [26]:
pricing_df["pricing_review_flag"] = (
    (pricing_df["price_vs_segment_pct"] > upper_limit) &
    (pricing_df["segment_size"] >= 100)
)

In [27]:
pricing_df["pricing_review_flag"].value_counts()

pricing_review_flag
False    93299
True      3267
Name: count, dtype: int64

In [28]:
eligible_pricing_df = pricing_df[
    pricing_df["segment_size"] >= 100
]

review_pct = (
    eligible_pricing_df["pricing_review_flag"].mean() * 100
)

print("Eligible listings:", len(eligible_pricing_df))
print("Review candidates:", eligible_pricing_df["pricing_review_flag"].sum())
print("Review %:", review_pct)

Eligible listings: 86865
Review candidates: 3267
Review %: 3.7610084614056296


In [29]:
pricing_features = pricing_df[
    [
        "id",
        "segment_median_price_m2",
        "price_vs_segment_pct",
        "segment_size",
        "pricing_review_flag"
    ]
].copy()

In [30]:
powerbi_df = powerbi_df.merge(
    pricing_features,
    on="id",
    how="left"
)

In [31]:
print(powerbi_df.shape)
print(powerbi_df["id"].nunique())

(121442, 33)
121442


In [32]:
powerbi_df["pricing_status"] = "Not eligible"

powerbi_df.loc[
    powerbi_df["segment_size"].notna() &
    (powerbi_df["segment_size"] < 100),
    "pricing_status"
] = "Insufficient comparables"

powerbi_df.loc[
    powerbi_df["segment_size"] >= 100,
    "pricing_status"
] = "Within expected range"

powerbi_df.loc[
    powerbi_df["pricing_review_flag"] == True,
    "pricing_status"
] = "Review candidate"

In [33]:
powerbi_df["pricing_status"].value_counts()

pricing_status
Within expected range       83598
Not eligible                29432
Insufficient comparables     5145
Review candidate             3267
Name: count, dtype: int64

In [34]:
powerbi_df.columns.tolist()

['id',
 'ad_type',
 'start_date',
 'end_date',
 'created_on',
 'lat',
 'lon',
 'l1',
 'l2',
 'l3',
 'l4',
 'l5',
 'l6',
 'rooms',
 'bedrooms',
 'bathrooms',
 'surface_total',
 'surface_covered',
 'price',
 'currency',
 'price_period',
 'title',
 'description',
 'property_type',
 'operation_type',
 'end_date_clean',
 'listing_duration_days',
 'price_per_m2',
 'price_range',
 'segment_median_price_m2',
 'price_vs_segment_pct',
 'segment_size',
 'pricing_review_flag',
 'pricing_status']

In [35]:
final_columns = [
    "id",
    "start_date",
    "end_date_clean",
    "lat",
    "lon",
    "l3",
    "rooms",
    "bedrooms",
    "bathrooms",
    "surface_total",
    "surface_covered",
    "price",
    "price_per_m2",
    "price_range",
    "listing_duration_days",
    "segment_median_price_m2",
    "price_vs_segment_pct",
    "segment_size",
    "pricing_review_flag",
    "pricing_status",
    "title"
]

final_df = powerbi_df[final_columns].copy()

In [36]:
final_df = final_df.rename(
    columns={
        "l3": "neighborhood",
        "end_date_clean": "end_date"
    }
)

In [37]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121442 entries, 0 to 121441
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   id                       121442 non-null  str           
 1   start_date               121442 non-null  datetime64[us]
 2   end_date                 91396 non-null   datetime64[us]
 3   lat                      112359 non-null  float64       
 4   lon                      112301 non-null  float64       
 5   neighborhood             116403 non-null  str           
 6   rooms                    112535 non-null  float64       
 7   bedrooms                 76150 non-null   float64       
 8   bathrooms                115275 non-null  float64       
 9   surface_total            96754 non-null   float64       
 10  surface_covered          98066 non-null   float64       
 11  price                    121442 non-null  float64       
 12  price_per_m2             96

In [38]:
numeric_columns = [
    "price_per_m2",
    "segment_median_price_m2",
    "price_vs_segment_pct"
]

final_df[numeric_columns] = final_df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

In [39]:
final_df[numeric_columns].dtypes

price_per_m2               float64
segment_median_price_m2    float64
price_vs_segment_pct       float64
dtype: object

In [40]:
print("Shape:", final_df.shape)
print("Unique IDs:", final_df["id"].nunique())
print("Duplicate IDs:", final_df["id"].duplicated().sum())
print("Review candidates:", (final_df["pricing_status"] == "Review candidate").sum())

Shape: (121442, 21)
Unique IDs: 121442
Duplicate IDs: 0
Review candidates: 3267


In [41]:
final_df.to_csv(
    "../data/processed/caba_apartments_final.csv",
    index=False
)

In [42]:
check_df = pd.read_csv(
    "../data/processed/caba_apartments_final.csv"
)

print(check_df.shape)
check_df.head()

(121442, 21)


,id,start_date,end_date,lat,lon,neighborhood,rooms,bedrooms,bathrooms,surface_total,...,price,price_per_m2,price_range,listing_duration_days,segment_median_price_m2,price_vs_segment_pct,segment_size,pricing_review_flag,pricing_status,title
0,/F4YPH2nVSaoiyunrdNBEQ==,2019-09-15,NaN,-34.572445,-58.420624,Palermo,4.0,NaN,3.0,193.0,...,935000.0,4844.559585,>500k,NaN,3235.294118,49.740933,2544.0,False,Within expected range,Departamento - Palermo
1,wb722Ak53RN+qPDHNnNiLw==,2019-09-15,NaN,-34.578547,-58.430038,Palermo,2.0,NaN,NaN,31.0,...,89000.0,2870.967742,<100k,NaN,3253.968254,-11.770260,4514.0,False,Within expected range,Departamento venta
2,CWhyNLsxy+1ixo/cDUN1Dg==,2019-09-15,NaN,-34.586423,-58.414190,Palermo,2.0,NaN,1.0,32.0,...,132900.0,4153.125000,100k-150k,NaN,3253.968254,27.632622,4514.0,False,Within expected range,Venta - Departamento 2 Amb. - Palermo
3,OtiAJtGJ6lH88FtDpGYhgw==,2019-09-15,2019-09-25,-34.584758,-58.413319,Palermo,3.0,NaN,1.0,88.0,...,230000.0,2613.636364,200k-300k,10.0,3152.173913,-17.084639,4043.0,False,Within expected range,Departamento - Palermo
4,6K4VVHqJpFMhuAD7O0oTXA==,2019-09-15,2019-09-30,-34.578491,-58.431951,Palermo,2.0,NaN,1.0,36.0,...,108000.0,3000.000000,100k-150k,15.0,3253.968254,-7.804878,4514.0,False,Within expected range,Departamento - Palermo
